In [1]:
import numpy as np
import os.path as op

from nilearn.connectome import ConnectivityMeasure
from brainspace.gradient import GradientMaps
from brainspace.utils.parcellation import map_to_labels
import numpy as np
import nibabel as nib
from nilearn import datasets
import os.path as op
import os
from nilearn import signal
import pandas as pd
from scipy.sparse.csgraph import connected_components
from utils import fsavTofsav5,cleanTS

bids_folder = '/Volumes/mrenkeED/data/ds-stressrisk'

240125-10:26:44,186 nipype.utils WARNING:
	 A newer version (1.8.4) of nipy/nipype is available. You are using 1.8.3


In [5]:
sub = '01'
ses = 1


In [6]:
atlas = datasets.fetch_atlas_surf_destrieux()
regions = atlas['labels'].copy()
masked_regions = [b'Medial_wall', b'Unknown']
masked_labels = [regions.index(r) for r in masked_regions]
for r in masked_regions:
    regions.remove(r)
labeling = np.concatenate([atlas['map_left'], atlas['map_right']])
labeling_noParcel = np.arange(0,len(labeling),1,dtype = int)     # Map gradients to original parcels
mask = ~np.isin(labeling, masked_labels) # generate a new, raw mask for each sub, that can be worked on later


In [8]:
space='fsnative'
runs = range(1, 7)#,space = 'fsaverage5', bids_folder='/Users/mrenke/data/ds-stressrisk'):
# load in data as timeseries and regress out confounds (for each run sepeprately)
number_of_vertex = 20484  # 'fsaverage5', 10242 * 2

fmriprep_confounds_include = ['global_signal', 'dvars', 'framewise_displacement', 'trans_x',
                            'trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z',
                            'a_comp_cor_00', 'a_comp_cor_01', 'a_comp_cor_02', 'a_comp_cor_03', 'cosine00', 'cosine01', 'cosine02'
                            ] # 

clean_ts_runs = np.empty([number_of_vertex,0])

ex_file = op.join(bids_folder,'derivatives', 'fmriprep', f'sub-{sub}', f'ses-{ses}', 'func', 
    f'sub-{sub}_ses-{ses}_task-risk_run-1_space-{space}_hemi-L_bold.func.gii')



In [9]:
timeseries = [None] * 2

for i, hemi in enumerate(['L', 'R']):
    ex_file = op.join(bids_folder,'derivatives', 'fmriprep', f'sub-{sub}', f'ses-{ses}', 'func', 
    f'sub-{sub}_ses-{ses}_task-risk_run-1_space-{space}_hemi-L_bold.func.gii')

    timeseries[i] = nib.load(ex_file).agg_data()
timeseries = np.vstack(timeseries) # (20484, 135)

In [12]:
timeseries.shape[0]

257086

In [14]:
number_of_vertex = timeseries.shape[0]

    # loop over runs and concatenate timeseries
clean_ts_runs = np.empty([number_of_vertex,0])
for run in runs:
    timeseries = [None] * 2
    for i, hemi in enumerate(['L', 'R']):
        filename = op.join(bids_folder,'derivatives', 'fmriprep', f'sub-{sub}', f'ses-{ses}', 'func', 
        f'sub-{sub}_ses-{ses}_task-risk_run-{run}_space-{space}_hemi-{hemi}_bold.func.gii')
        
    
        timeseries[i] = nib.load(filename).agg_data()
    timeseries = np.vstack(timeseries) # (20484, 135)
    print(timeseries.shape)

    fmriprep_confounds_file = op.join(bids_folder,'derivatives', 'fmriprep', f'sub-{sub}', f'ses-{ses}', 'func', f'sub-{sub}_ses-{ses}_task-risk_run-{run}_desc-confounds_timeseries.tsv')
    fmriprep_confounds = pd.read_table(fmriprep_confounds_file)[fmriprep_confounds_include] 
    fmriprep_confounds= fmriprep_confounds.fillna(method='bfill')

    #clean_ts_list[run] = signal.clean(timeseries.T, confounds=fmriprep_confounds).T
    clean_ts = signal.clean(timeseries.T, confounds=fmriprep_confounds).T

    clean_ts_runs = np.append(clean_ts_runs, clean_ts, axis=1)



(256422, 135)
(256422, 135)
(256422, 135)
(256422, 135)
(256422, 135)
(256422, 135)


In [15]:
clean_ts_runs.shape

(256422, 810)

In [18]:
seed_ts_noParcel = clean_ts_runs
correlation_measure_noParcel = ConnectivityMeasure(kind='correlation')


In [19]:
graph = correlation_measure_noParcel.fit_transform([seed_ts_noParcel.T])[0] #correlation_matrix_noParcel
cc = connected_components(graph)
mask_cc = cc[1] == 0 # all nodes in 0 belong to the largest connected component, check #-components in cc[0]


KeyboardInterrupt: 